# Pulser on Pasqal through qBraid

This notebook runs [Pulser](https://pulser.readthedocs.io/) pulse sequences on [Pasqal](https://www.pasqal.com/)'s neutral-atom devices through the `QbraidProvider`. You only need a qBraid API key: jobs are billed in qBraid credits, and the `emu-free` emulator used here is free.

Pasqal's devices run *analog* programs. Instead of applying gates, you place atoms and shape the laser pulses that drive them. We use that to show the **Rydberg blockade**: two atoms close together can't both be excited.

In [ ]:
%%capture

%pip install 'qbraid[pulser,visualization]'

## Pasqal devices on qBraid

Every Pasqal device ID starts with `pasqal:`. `emu-free` is free; the other emulators and the Fresnel QPU are billed per minute of execution time.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()

for device in provider.get_devices():
    if device.id.startswith("pasqal:"):
        print(f"{device.id:34} {device.profile.num_qubits:>3} atoms  {device.status().name}")

## Build a sequence

A Pulser sequence has a *register*, the atom positions in µm, and the pulses applied through the device's channels. This one places two atoms `distance_um` apart and applies a global pulse whose area, π, takes a lone atom from its ground state to its Rydberg state. It ends by measuring each atom: `1` if it was excited, `0` if not.

In [ ]:
import numpy as np
from pulser import BlackmanWaveform, Pulse, Register, Sequence
from pulser.devices import AnalogDevice


def two_atoms(distance_um):
    register = Register.from_coordinates([(0, 0), (distance_um, 0)], prefix="q")
    sequence = Sequence(register, AnalogDevice)
    sequence.declare_channel("rydberg", "rydberg_global")
    pulse = Pulse.ConstantDetuning(BlackmanWaveform(1000, np.pi), detuning=0, phase=0)
    sequence.add(pulse, "rydberg")
    sequence.measure("ground-rydberg")
    return sequence


two_atoms(20).draw()

## Run on the free emulator

With the atoms 20 µm apart, they barely interact, so the pulse excites both: expect `'11'` almost every shot.

In [ ]:
device = provider.get_device("pasqal:pasqal:sim:emu-free")

far_job = device.run(two_atoms(20), shots=200)
far_job.status()

In [ ]:
far_counts = far_job.result().data.get_counts()
far_counts

## The Rydberg blockade

Now move the atoms to 5 µm apart. An excited atom shifts its neighbour's Rydberg level far out of resonance, so the two can't both be excited: `'11'` disappears, and a single excitation is shared between the atoms instead. The blockaded pair responds √2 times faster than a lone atom, so the same pulse over-rotates it, and some shots end back in `'00'`.

In [ ]:
near_job = device.run(two_atoms(5), shots=200)

near_counts = near_job.result().data.get_counts()
near_counts

In [ ]:
from qbraid.visualization import plot_histogram

plot_histogram([far_counts, near_counts], legend=["20 µm apart", "5 µm apart"])

## Add noise with a runtime option

Pasqal's emulators take a `backend_configuration` runtime option: a Pulser `EmulationConfig`, passed as its abstract representation. Here it adds measurement errors: a 5% chance of reading a ground-state atom as excited, and 10% the other way.

In [ ]:
from pulser.backend import BitStrings, EmulationConfig
from pulser.noise_model import NoiseModel

config = EmulationConfig(
    observables=[BitStrings()],
    noise_model=NoiseModel(p_false_pos=0.05, p_false_neg=0.1),
)

noisy_job = device.run(
    two_atoms(20),
    shots=200,
    runtime_options={"backend_configuration": config.to_abstract_repr()},
)
noisy_job.result().data.get_counts()

## When Pasqal rejects a job

A job Pasqal can't run ends `FAILED`, costs nothing, and carries Pasqal's reason. `emu-free` takes at most 12 atoms, so a 13-atom register fails:

In [ ]:
big_register = Register.from_coordinates([(6 * i, 0) for i in range(13)], prefix="q")
big_sequence = Sequence(big_register, AnalogDevice)
big_sequence.declare_channel("rydberg", "rydberg_global")
big_sequence.add(Pulse.ConstantDetuning(BlackmanWaveform(1000, np.pi), 0, 0), "rydberg")
big_sequence.measure("ground-rydberg")

failed_job = device.run(big_sequence, shots=100)
failed_job.wait_for_final_state()

status = failed_job.status()
print(status, status.status_message)

## Next steps

- Run larger registers on `pasqal:pasqal:sim:emu-mps` (80 atoms) or `pasqal:pasqal:sim:emu-tn` (100 atoms), billed per minute of execution time.
- Test a QPU sequence on `pasqal:pasqal:sim:emu-fresnel` before running it on the Fresnel QPU, `pasqal:pasqal:qpu:fresnel-can1`.
- Read the [Pasqal guide](https://docs.qbraid.com/v2/sdk/user-guide/providers/pasqal) in the qBraid docs and the [Pulser tutorials](https://pulser.readthedocs.io/).

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>